### Performance evaluations: Precision - Recall and Average Percision ###

In [2]:
import sys
import os
import numpy as np
import json
import copy
import pandas as pd
import logging
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
from sklearn import metrics
import seaborn as sns

logger = logging.getLogger(__name__)

# PyTorch
import torch
from torchvision import ops

%load_ext autoreload
%autoreload 2
import computervision
from computervision.imageproc import is_image, ImageData, clipxywh, xyxy2xywh, xywh2xyxy, plot_boxes
from computervision.datasets import DETRdataset
from computervision.transformations import AugmentationTransform
from computervision.performance import DetectionMetrics
from computervision.inference import DETRinference, get_gpu_info

print(f'Project version: {computervision.__version__}')
print(f'Authors: {computervision.__authors__}')
print(f'Python version:  {sys.version}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project version: v0.0.2
Authors: The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]


In [3]:
# Check GPU availability
device, device_str = get_gpu_info()

CUDA available: True
Number of GPUs found:  1
Current device ID: 0
GPU device name:   NVIDIA GeForce RTX 3060 Laptop GPU
PyTorch version:   2.8.0a0+34c6371d24.nv25.08
CUDA version:      13.0
CUDNN version:     91200
Current device:    cuda:0


### Test data ###

In [4]:
# Dentex test data
data_dir = os.environ.get('DATA')
dataset_name = 'dataset_object_dentex_251011'
image_dir = os.path.join(data_dir, dataset_name, 'test')
test_df_file_name = 'dataset_object_dentex_251011_test.parquet'
test_df_file = os.path.join(image_dir, test_df_file_name)
df = pd.read_parquet(test_df_file)

# Filter the data frame
dset_col = 'dset'
pos_col = 'ada'
file_col = 'file_name'
bbox_col = 'bbox'
df = df.loc[(df[dset_col] == 'test') & (df['transformation'] == 5)]
display(df.head(2))
print(f'Images in test data: {len(df[file_col].unique())}')
print(f'Annotations:         {df.shape[0]}')

,bbox,quadrant,ada,file_name,file_base_name,quadrants,height,width,transformation,transformation_name,dset
10256,"[572, 228, 67, 338]",1,8,test_train_219_01_05.png,train_219,1,640,640,5,test_set,test
10257,"[511, 234, 91, 332]",1,7,test_train_219_01_05.png,train_219,1,640,640,5,test_set,test


Images in test data: 160
Annotations:         1275


### Model ###

In [5]:
model_name = 'rtdetr_dtx_hsdm_251014_01'
checkpoint = 6400
model_dir = os.path.join(data_dir, 'model', model_name)
checkpoint_dir = os.path.join(model_dir, f'checkpoint-{checkpoint}')
model_config_file = os.path.join(model_dir, f'{model_name}.json')
with open(model_config_file, mode='r') as file:
    model_config = json.load(file)
display(*list(model_config.keys()), sep='\n')

# Load the model for inference
dtr = DETRinference(device_name='cuda:0', 
                    checkpoint_path=checkpoint_dir, 
                    batch_size = 16)

'model_info'

'id2label'

'training_args'

'processor_params'

'bbox_format'

### Predict bounding boxes on the test data ###

In [6]:
# Create a PyTorch data set
transforms = AugmentationTransform().get_transforms(name='val')
bbox_format = model_config.get('bbox_format')
dataset = DETRdataset(data=df.copy(), 
                      image_processor=dtr.processor, 
                      image_dir=image_dir, 
                      file_name_col=file_col, 
                      label_id_col=None, 
                      bbox_col=None, 
                      bbox_format=bbox_format, 
                      transforms=transforms)
print(f'Total images in test data: {len(dataset)}')
# Run the forward pass to get the predictions
threshold = 0.05
pred_raw = dtr.predict_on_dataset(dataset, threshold=threshold)

Total images in test data: 160
Predicting batch 10 of 10.


In [7]:
# Make a copy of the predictions 
pred = copy.deepcopy(pred_raw)

# Add the file names to the data frame
file_names = df[file_col].unique()
id2file = dict(zip(range(len(file_names)), file_names))
pred[file_col] = pred['image_id'].apply(lambda image_id: id2file.get(image_id))

# Add the label names (the positions) to the data frame
id2label = {int(category_id): int(label) for category_id, label in model_config.get('id2label').items()}
pred[pos_col] = pred['category_id'].apply(lambda category_id: id2label.get(category_id))

print(f'Images in output data:     {len(pred['image_id'].unique())}')
display(pred.head(2))

# Let's find out if all of the images resulted in predictions
pred_empty = len(pred.loc[pred[pos_col].isnull(), file_col].unique())

# Filter out rows with images that do not have predictions
pred = pred.loc[~pred[pos_col].isnull()]
print(f'Total number of images in data set:             {len(dataset)}')
print(f'Images without predictions for threshold ({threshold}):{pred_empty}')
print(f'Images with predictions:                        {len(pred[file_col].unique())}')

Images in output data:     160


,image_id,image_width,image_height,batch,category_id,bbox,score,area,file_name,ada
0,0,640,640,0,5,"[403, 237, 230, 328]",0.133645,75440,test_train_219_01_05.png,6
1,0,640,640,0,2,"[116, 186, 270, 344]",0.115267,92880,test_train_219_01_05.png,3


Total number of images in data set:             160
Images without predictions for threshold (0.05):0
Images with predictions:                        160


### Classify predictions: TP, FP, FN ###
We create a method to classify all of the predictions in the data set. 

In [112]:
true_df = copy.deepcopy(df)
pred_df = copy.deepcopy(pred)
bbox_col = 'bbox'
label_col = 'ada'
file_col = 'file_name'
score_col = 'score'
iou_threshold = 0.5

# def precision_recall(true_df, pred_df, file_col, label_col, bbox_col, score_col, iou_threshold):
# We want to use only images with predictions
pred_df = pred_df.loc[~pred_df[label_col].isnull()]

# And the file names in the two input data frame must match
file_list = sorted(list(set(true_df[file_col].tolist()).\
    intersection(pred_df[file_col].tolist())))

classifications_df_list = []
missed_df_list = []

for f, file in enumerate(file_list):
    true_bboxes = true_df.loc[true_df[file_col] == file, bbox_col].tolist()
    pred_bboxes = pred_df.loc[pred_df[file_col] == file, bbox_col].tolist()
    
    true_bboxes = [list(np.int64(box)) for box in true_bboxes]
    pred_bboxes = [list(np.int64(box)) for box in pred_bboxes]
    
    true_labels = true_df.loc[true_df[file_col] == file, label_col].tolist()
    pred_labels = pred_df.loc[pred_df[file_col] == file, label_col].tolist()
    pred_scores = pred_df.loc[pred_df[file_col] == file, score_col].tolist()

    pred_cl = DetectionMetrics().\
        classify_predictions(true_labels=true_labels, 
                             true_bboxes=true_bboxes, 
                             pred_labels=pred_labels, 
                             pred_bboxes=pred_bboxes, 
                             iou_threshold=iou_threshold).\
        rename(columns={'pred_label': label_col})

    # Add more information about the predictions to the output
    pred_cl.insert(loc=0, column=file_col, value=file)
    pred_cl.insert(loc=1, column='iou_threshold', value=iou_threshold)
    pred_cl.insert(loc=2, column=score_col, value=pred_scores)
    pred_cl.insert(loc=3, column=bbox_col, value=pred_bboxes)

    classifications_df_list.append(pred_cl)
    
    # False negatives: Labels in the ground truth data that were not detected
    missed_label_list = sorted(list(set(true_labels).difference(pred_labels)))

    if len(missed_label_list) > 0:
        missed_cl = pd.DataFrame({label_col: missed_label_list})
        missed_cl.insert(loc=0, column=file_col, value=file)
        missed_df_list.append(missed_cl)

if len(cl_df_list) > 0:
    classifications = pd.concat(classifications_df_list, axis=0, ignore_index=True)
    classifications = classifications.\
        sort_values(by=[label_col, score_col], ascending=True).\
        reset_index(drop=True)
else:
    classifications = None

if len(missed_df_list) > 0:
    missed = pd.concat(missed_df_list, axis=0, ignore_index=True)
    missed = missed.\
        sort_values(by=label_col, ascending=True).\
        reset_index(drop=True)

# Calculate the missed predictions for each file
n_missed = missed.\
    groupby(file_col).\
    count().\
    reset_index(drop=False).\
    rename(columns={label_col: 'n_missed'}).\
    sort_values(by='n_missed', ascending=False).\
    reset_index(drop=True)

### Precision - recall for each label ###

In [161]:
pr_df_list = []
label_list = sorted(list(classifications[label_col].unique()))

for label in label_list:

    # Total number of positives in the data set (TP + FN)
    n_labels = len(true_df.loc[true_df[label_col] == label])
    
    # Predictions for this class sorted by score in descending order
    classifications_label = classifications.\
        loc[classifications[label_col] == label].\
        sort_values(by='score', ascending=False).\
        reset_index(drop=True)
    
    # Calculate precision and recall for each row
    correct = classifications_label['TP'].tolist()
    
    # precision = true positives / all detections
    precision=[sum(correct[:i + 1])/(i + 1) for i in range(len(correct))]
    
    # recall = true positives / samples with this label in ground truth data
    recall=[sum(correct[:i + 1])/n_labels for i in range(len(correct))]
    
    # Add precision and recall to the data frame for this label
    classifications_label = classifications_label.\
        assign(precision=precision, recall=recall)
    
    # Calculate precision and recall independent from the bounding box
    # We count every prediction that is in the image as positive
    # Detections that were not in the image did not get an iou value (FP)
    
    # TP + FP
    n_detections = len(classifications_label)
    # TP: all detections for that class with a ground truth label, so IoU >= 0
    n_detections_with_iou = len(classifications_label.loc[~classifications_label['IoU'].isnull()])
    # We can add a precision and recall value that is just for this class, indepdendent from the bounding box
    precision_label = n_detections_with_iou / n_detections
    recall_label = n_detections_with_iou / n_labels

    # Calculate the AUC
    auc = metrics.auc(x=recall, y=precision)
    
    # Add the class-level precision/recall values to the data frame
    classifications_label = classifications_label.\
        assign(precision_label=precision_label, 
               recall_label=recall_label,
               auc=auc)
    
    pr_df_list.append(classifications_label)

pr_df = pd.concat(pr_df_list, axis=0, ignore_index=True)

auc_df = pr_df[[label_col, 'auc']].\
    sort_values(by=label_col, ascending=True).\
    groupby(by=label_col).first().reset_index(drop=False)

### Visualizations ###

In [162]:
pr_df

,file_name,iou_threshold,score,bbox,ada,TP,IoU,n_missed,duplicate_TP,precision,recall,precision_label,recall_label,auc
0,test_train_415_01_05.png,0.5,0.156303,"[68, 170, 208, 312]",1,0,NaN,0,False,0.0,0.0,0.21875,0.583333,0.009479
1,test_train_254_01_05.png,0.5,0.143298,"[7, 149, 257, 378]",1,0,0.261577,0,False,0.0,0.0,0.21875,0.583333,0.009479
2,test_train_287_01_05.png,0.5,0.132561,"[45, 188, 126, 355]",1,0,NaN,3,False,0.0,0.0,0.21875,0.583333,0.009479
3,test_train_618_01_05.png,0.5,0.120589,"[95, 183, 82, 302]",1,0,NaN,1,False,0.0,0.0,0.21875,0.583333,0.009479
4,test_train_564_01_05.png,0.5,0.117467,"[43, 199, 182, 348]",1,0,NaN,0,False,0.0,0.0,0.21875,0.583333,0.009479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2536,test_train_415_12_05.png,0.5,0.052792,"[0, 10, 637, 163]",32,0,NaN,4,False,0.0,0.0,0.00000,0.000000,0.000000
2537,test_train_226_12_05.png,0.5,0.052778,"[10, 0, 630, 260]",32,0,NaN,5,False,0.0,0.0,0.00000,0.000000,0.000000
2538,test_train_564_23_05.png,0.5,0.050748,"[97, 1, 151, 295]",32,0,NaN,2,False,0.0,0.0,0.00000,0.000000,0.000000
2539,test_train_285_23_05.png,0.5,0.050388,"[202, 288, 232, 352]",32,0,NaN,4,False,0.0,0.0,0.00000,0.000000,0.000000
